# Notebook 1: QSARmil multi-conformer modelling

This notebook introduces the high-level API for building multi-conformer models with **QSARmil**. The API is designed to simplify benchmarking QSARmil against alternative approaches without requiring detailed knowledge of the underlying modelling configuration. As a result, the workflow remains concise and easy to follow. You can adapt the code below to your own dataset and run the complete **QSARmil** modelling pipeline with minimal modifications.

In [ ]:
import pandas as pd

from sklearn.metrics import r2_score

### 1. Data load

As an example, we use a publicly available and easily accessible collection of molecular bioactivity datasets introduced in the paper by:

> Van Tilborg, Derek, Alisa Alenicheva, and Francesca Grisoni. "Exposing the limitations of molecular machine learning with activity cliffs." Journal of chemical information and modeling 62.23 (2022): 5938-5951.

From this collection, we select one dataset for demonstration purposes.

In [ ]:
# load data
url = "https://raw.githubusercontent.com/molML/MoleculeACE/main/MoleculeACE/Data/benchmark_data/CHEMBL2034_Ki.csv"
df_ace = pd.read_csv(url)

# train/test split
df_train = df_ace[df_ace["split"] == "train"][["smiles", "y"]]
df_test = df_ace[df_ace["split"] == "test"][["smiles", "y"]]

In [ ]:
# uncomment for quick testing
# df_train = df_train.sample(frac=0.1, random_state=42).reset_index(drop=True)
# df_test = df_test.sample(frac=0.1, random_state=42).reset_index(drop=True)
# df_train.shape, df_test.shape

### 2. Data validation

The data validation step is straightforward but includes an additional requirement compared to standard 2D modelling pipelines. In most QSAR workflows, input structures are primarily validated by checking whether their SMILES strings are valid and can be successfully parsed into molecular objects.

In contrast, **QSARmil** introduces an extra validation step: verifying whether a valid 3D structure can be generated for each molecule. Molecules that fail this step cannot be used in the 3D modelling pipeline. As a result, the filtering (removal) rate in QSARmil may be higher than in typical 2D QSAR workflows.

In [ ]:
from qsarmil.data.input_data import DataValidator

In [ ]:
dvalid = DataValidator(num_cpu=30, verbose=True)

df_train = dvalid.filter_dataframe(df_train)
df_test = dvalid.filter_dataframe(df_test)

### 3. Building model

The multi-conformer model building pipeline consists of several sequential steps:

 - Conformer generation: the maximum number of conformers is defined using the ``num_conf`` parameter.
 - Descriptor calculation: by default, multiple types of 3D molecular descriptors are computed.
 - Model training: several multi-instance learning networks are used to train models. Internal stepwise hyperparameter optimization can be enabled with ``hopt=True`` (note that this increases runtime).
 - Consensus search: once multiple multi-conformer models are trained, a genetic algorithm is applied to identify an optimal consensus model.

Task type is defined automatically (regression or binary classification), ``output_folder`` (directory for storing predictions from individual models), and ``verbose`` (controls the level of pipeline progress output).

In [ ]:
from qsarmil.meta import MultiConformerModel

In [ ]:
model = MultiConformerModel(num_conf=10, hopt=False, verbose=True, output_folder="./mcfm")
df_pred = model.run_predict(df_train, df_test)

In [ ]:
r2_score(df_test["y"], df_pred["pred"])

In [ ]:
model.save()

In [ ]:
preds = model.predict(df_test)
preds

In [ ]:
r2_score(df_test["y"], preds["pred"])

In [ ]:
model = MultiConformerModel.load("./mcfm/model.pkl")
model.predictFromSMILES(df_test["smiles"])

In [ ]:
# works on unknown SMILES as well, no caching in that case
model.predictFromSMILES(
    [
        "CN1[C@H]2CC[C@@H]1[C@@H](C(OC)=O)[C@@H](OC(C3=CC=CC=C3)=O)C2",
        "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O"
    ]
)